# Clustering — Introduction and Types

**Clustering** is *unsupervised* learning: group data points so that points in the same cluster are similar and
points in different clusters are dissimilar — **without labels**.

Typical uses: customer segmentation, document/topic grouping, image segmentation, anomaly detection, data exploration.

## Types of clustering algorithms

| Type | Idea | Cluster shape | Examples | Notebook |
|---|---|---|---|---|
| **Partitioning (centroid-based)** | Split data into $k$ groups, each represented by a centre; minimise within-cluster distance | Convex, roughly spherical | K-Means, K-Medoids | `15_KMeans` |
| **Hierarchical** | Build a tree (dendrogram) of nested clusters by merging (agglomerative) or splitting (divisive) | Depends on linkage | Single / complete / average / Ward linkage | `16_Hierarchical_Clustering` |
| **Density-based** | Clusters are dense regions separated by sparse regions; finds noise | Arbitrary | DBSCAN, OPTICS, HDBSCAN | `17_DBSCAN` |
| **Distribution (model)-based** | Data generated by a mixture of probability distributions | Elliptical | Gaussian Mixture Models (EM) | — |
| **Grid-based** | Quantise space into cells and cluster dense cells | Axis-aligned regions | STING, CLIQUE | — |

**Hard** clustering assigns each point to exactly one cluster; **soft** (fuzzy) clustering gives membership degrees.

## Key ingredients
1. **A similarity / distance measure** (Euclidean, Manhattan, cosine, ...)
2. **An algorithm** (above)
3. **Evaluation**
   * *Internal* (no labels): inertia / within-cluster SSE, silhouette score, Davies–Bouldin
   * *External* (labels known, e.g. for benchmarking): purity, Rand index

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_circles(n_samples=200, factor=0.5, noise=0.05, random_state=None):
    """A large circle containing a smaller circle."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = rng.uniform(0, 2 * np.pi, n_out)
    t_in = rng.uniform(0, 2 * np.pi, n_in)
    X = np.vstack([np.c_[np.cos(t_out), np.sin(t_out)],
                   factor * np.c_[np.cos(t_in), np.sin(t_in)]])
    X += rng.normal(0, noise, size=X.shape)
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

## Distance measures

In [ ]:
def euclidean(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def manhattan(a, b):
    return np.sum(np.abs(a - b))

def chebyshev(a, b):
    return np.max(np.abs(a - b))

def cosine_distance(a, b):
    return 1 - a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

a, b = np.array([1.0, 2.0]), np.array([4.0, 6.0])
for f in [euclidean, manhattan, chebyshev, cosine_distance]:
    print(f"{f.__name__:16s}: {f(a, b):.4f}")

In [ ]:
# unit "circles" of each distance: the set of points at distance 1 from the origin
t = np.linspace(0, 2 * np.pi, 400)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, p, name in zip(axes, [1, 2, np.inf], ["Manhattan (L1)", "Euclidean (L2)", "Chebyshev (L∞)"]):
    pts = np.c_[np.cos(t), np.sin(t)]
    norms = np.max(np.abs(pts), 1) if p == np.inf else (np.abs(pts) ** p).sum(1) ** (1 / p)
    pts = pts / norms[:, None]
    ax.plot(pts[:, 0], pts[:, 1]); ax.set_aspect("equal"); ax.set_title(name); ax.grid(alpha=0.3)
plt.suptitle("Points at distance 1 from the origin"); plt.tight_layout(); plt.show()

## Evaluation measures

In [ ]:
def pairwise_distances(A, B):
    sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.maximum(sq, 0))


def silhouette_score(X, labels):
    """Mean of s(i) = (b - a) / max(a, b); a = mean intra-cluster distance,
    b = mean distance to the nearest other cluster. Noise points (label -1) are ignored."""
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    clusters = np.unique(labels)
    if len(clusters) < 2:
        return np.nan
    D = pairwise_distances(X, X)
    s = np.zeros(len(X))
    for i in range(len(X)):
        same = labels == labels[i]
        if same.sum() == 1:
            continue                                   # singleton cluster: s(i) = 0
        a = D[i, same].sum() / (same.sum() - 1)
        b = min(D[i, labels == c].mean() for c in clusters if c != labels[i])
        s[i] = (b - a) / max(a, b)
    return s.mean()


def inertia(X, labels):
    """Within-cluster sum of squared distances to the cluster mean (what K-Means minimises)."""
    return sum(((X[labels == c] - X[labels == c].mean(0)) ** 2).sum() for c in np.unique(labels))


def purity(y_true, labels):
    """Fraction of points belonging to the majority true class of their cluster."""
    return sum(np.bincount(y_true[labels == c]).max() for c in np.unique(labels)) / len(y_true)


def rand_index(y_true, labels):
    """Fraction of point pairs on which the two partitions agree (same/same or different/different)."""
    same_true = y_true[:, None] == y_true[None, :]
    same_pred = labels[:, None] == labels[None, :]
    iu = np.triu_indices(len(y_true), 1)
    return np.mean(same_true[iu] == same_pred[iu])

In [ ]:
X, y = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=3)
rng = np.random.RandomState(0)
random_labels = rng.randint(0, 3, len(X))

print(f"{'':16s}{'inertia':>10s}{'silhouette':>12s}{'purity':>8s}{'rand':>8s}")
for name, lab in [("true grouping", y), ("random grouping", random_labels)]:
    print(f"{name:16s}{inertia(X, lab):10.1f}{silhouette_score(X, lab):12.3f}{purity(y, lab):8.3f}{rand_index(y, lab):8.3f}")

## Why there are many algorithms: data shape matters
A tiny K-Means (centroid-based) handles compact blobs but fails on non-convex shapes, which density-based methods handle.

In [ ]:
def tiny_kmeans(X, k, n_iter=50, seed=0):
    rng = np.random.RandomState(seed)
    centers = X[rng.choice(len(X), k, replace=False)]
    for _ in range(n_iter):
        labels = np.argmin(((X[:, None] - centers[None]) ** 2).sum(-1), axis=1)
        centers = np.array([X[labels == j].mean(0) if np.any(labels == j) else centers[j] for j in range(k)])
    return labels

datasets = {
    "blobs": make_blobs(300, centers=3, cluster_std=1.0, random_state=3),
    "moons": make_moons(300, noise=0.06, random_state=0),
    "circles": make_circles(300, factor=0.45, noise=0.05, random_state=0),
    "anisotropic": (make_blobs(300, centers=3, random_state=170)[0] @ np.array([[0.6, -0.6], [-0.4, 0.8]]),
                    make_blobs(300, centers=3, random_state=170)[1]),
}
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col, (name, (Xd, yd)) in enumerate(datasets.items()):
    k = len(np.unique(yd))
    axes[0, col].scatter(Xd[:, 0], Xd[:, 1], c=yd, cmap="viridis", s=10); axes[0, col].set_title(f"{name}: true groups")
    lab = tiny_kmeans(Xd, k)
    axes[1, col].scatter(Xd[:, 0], Xd[:, 1], c=lab, cmap="viridis", s=10)
    axes[1, col].set_title(f"K-Means, purity={purity(yd, lab):.2f}")
plt.tight_layout(); plt.show()

## Summary
* Clustering discovers structure without labels; the "right" clusters depend on the distance measure and the algorithm's assumptions.
* **Partitioning** methods are fast and work for compact, similarly-sized clusters.
* **Hierarchical** methods give a full tree of clusterings — no need to fix $k$ in advance.
* **Density-based** methods find arbitrarily shaped clusters and flag noise.
* Always validate with internal measures (silhouette, inertia) and, when possible, domain knowledge.